# 04_Baseline_Per_Dataset_Comparison

Notebook này là bản baseline mới để so sánh dễ hơn với 05. Bản cũ không bị xóa. Điểm mới:

- Train/test classical ML trên **combined random**.
- Train/test classical ML trên **combined strict speaker-aware no TESS**.
- Train/test riêng từng dataset: CREMA-D, RAVDESS, SAVEE, TESS.
- Xuất bảng metric, per-dataset/per-class, confusion matrix và file zip.

Mục đích: sau này khi 05 dual-view/1D/2D chạy xong, ta có baseline cùng protocol để so sánh công bằng.


## 1. Imports and Paths


In [ ]:
from pathlib import Path
import os, json, time, random, shutil, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import librosa, soundfile as sf

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report, confusion_matrix
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC, SVC
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
import joblib

warnings.filterwarnings('ignore')
SEED = int(os.getenv('SEED', '42'))
random.seed(SEED); np.random.seed(SEED)

def display_table(x):
    try:
        display(x)
    except Exception:
        print(x)

def find_ser_processed():
    candidates = []
    env_path = os.getenv('SER_PROCESSED', '').strip()
    if env_path:
        candidates.append(Path(env_path))
    candidates.extend([
        Path('/kaggle/input/datasets/quanghuy225/ser-processed/ser_processed'),
        Path('/kaggle/input/ser-processed/ser_processed'),
        Path('/kaggle/input/ser-processed'),
        Path('/kaggle/working/ser_processed'),
        Path.cwd() / 'ser_processed',
        Path.cwd().parent / 'ser_processed',
        Path.cwd().parent / '01&02_Data_and_DataProcessing' / 'ser_processed',
        Path('D:/UTE/Speech Programming/Speech Project/01&02_Data_and_DataProcessing/ser_processed'),
    ])
    for candidate in candidates:
        if (candidate / 'metadata.csv').exists():
            return candidate.resolve()
    for root in [Path('/kaggle/input'), Path.cwd(), Path.cwd().parent]:
        if root.exists():
            for metadata_path in root.rglob('metadata.csv'):
                if metadata_path.parent.name == 'ser_processed':
                    return metadata_path.parent.resolve()
    raise FileNotFoundError('Cannot find ser_processed/metadata.csv')

SER_PROCESSED = find_ser_processed()
AUDIO_16K_DIR = SER_PROCESSED / 'audio_16k'
PROJECT_ROOT = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
OUTPUT_DIR = PROJECT_ROOT / '04_Baseline_Per_Dataset_outputs'
REPORT_DIR = OUTPUT_DIR / 'reports'; FIGURE_DIR = OUTPUT_DIR / 'figures'; MODEL_DIR = OUTPUT_DIR / 'models'; PRED_DIR = OUTPUT_DIR / 'predictions'; CACHE_DIR = OUTPUT_DIR / 'cache'
for d in [REPORT_DIR, FIGURE_DIR, MODEL_DIR, PRED_DIR, CACHE_DIR]:
    d.mkdir(parents=True, exist_ok=True)
print('SER_PROCESSED:', SER_PROCESSED)
print('OUTPUT_DIR:', OUTPUT_DIR)


## 2. Configuration


In [ ]:
QUICK_RUN = os.getenv('QUICK_RUN', '0') == '1'
QUICK_RUN_PER_GROUP = int(os.getenv('QUICK_RUN_PER_GROUP', '30'))
TARGET_SR = 16_000
TARGET_DURATION = float(os.getenv('TARGET_DURATION', '3.0'))
TARGET_LENGTH = int(TARGET_SR * TARGET_DURATION)
N_MFCC = 40
N_FFT = 400
WIN_LENGTH = 400
HOP_LENGTH = 160
COMMON_EMOTIONS = ['neutral', 'happy', 'sad', 'angry', 'fear', 'disgust']
LABEL_TO_ID = {label: i for i, label in enumerate(COMMON_EMOTIONS)}
ID_TO_LABEL = {i: label for label, i in LABEL_TO_ID.items()}

MODEL_SET = os.getenv('MODEL_SET', 'full').strip().lower()  # fast or full
RUN_COMBINED_RANDOM = os.getenv('RUN_COMBINED_RANDOM', '1') == '1'
RUN_COMBINED_STRICT = os.getenv('RUN_COMBINED_STRICT', '1') == '1'
RUN_SINGLE_DATASET = os.getenv('RUN_SINGLE_DATASET', '1') == '1'
CACHE_FEATURES = os.getenv('CACHE_FEATURES', '1') == '1'
print({'QUICK_RUN': QUICK_RUN, 'MODEL_SET': MODEL_SET, 'RUN_SINGLE_DATASET': RUN_SINGLE_DATASET})


## 3. Load Metadata and Build Comparable Splits


In [ ]:
metadata = pd.read_csv(SER_PROCESSED / 'metadata.csv')
metadata = metadata[metadata['emotion'].isin(COMMON_EMOTIONS)].copy()
metadata = metadata[metadata.get('readable', True) == True].copy()
metadata['label_id'] = metadata['emotion'].map(LABEL_TO_ID).astype(int)
metadata['speaker_id'] = metadata['speaker_id'].astype(str)
if QUICK_RUN:
    metadata = (metadata.sort_values(['dataset', 'emotion', 'sample_id'])
        .groupby(['dataset', 'emotion'], group_keys=False).head(QUICK_RUN_PER_GROUP).reset_index(drop=True))
else:
    metadata = metadata.reset_index(drop=True)

metadata['split_strict_original'] = metadata['split'].astype(str).str.lower() if 'split' in metadata.columns else 'train'
metadata.loc[~metadata['split_strict_original'].isin(['train', 'validation', 'test']), 'split_strict_original'] = 'train'
savee_plan = {'savee_DC': 'train', 'savee_JE': 'train', 'savee_JK': 'validation', 'savee_KL': 'test'}
metadata['split_strict_project'] = metadata['split_strict_original']
metadata.loc[metadata['dataset'].eq('SAVEE'), 'split_strict_project'] = metadata.loc[metadata['dataset'].eq('SAVEE'), 'speaker_id'].map(savee_plan).fillna('train')
metadata.loc[metadata['dataset'].eq('TESS'), 'split_strict_project'] = 'excluded'

tr, temp = train_test_split(metadata.index, test_size=0.30, random_state=SEED, stratify=metadata['label_id'])
va, te = train_test_split(temp, test_size=0.50, random_state=SEED, stratify=metadata.loc[temp, 'label_id'])
metadata['split_random'] = 'train'
metadata.loc[va, 'split_random'] = 'validation'
metadata.loc[te, 'split_random'] = 'test'

display_table(metadata.groupby(['dataset', 'emotion']).size().unstack(fill_value=0))
print('random:', metadata['split_random'].value_counts().to_dict())
print('strict:', metadata['split_strict_project'].value_counts().to_dict())


## 4. Feature Extraction for Classical ML


In [ ]:
def center_crop_or_pad(y, target_length=TARGET_LENGTH):
    if len(y) > target_length:
        start = (len(y) - target_length) // 2
        y = y[start:start + target_length]
    elif len(y) < target_length:
        pad = target_length - len(y)
        y = np.pad(y, (pad // 2, pad - pad // 2), mode='constant')
    return y.astype(np.float32)

def read_audio(row):
    cached = AUDIO_16K_DIR / f'{row.sample_id}.wav'
    if cached.exists():
        y, sr = sf.read(cached)
        if y.ndim > 1:
            y = y.mean(axis=1)
        if sr != TARGET_SR:
            y = librosa.resample(y.astype(np.float32), orig_sr=sr, target_sr=TARGET_SR)
    else:
        y, sr = librosa.load(row.filepath, sr=TARGET_SR, mono=True)
    return center_crop_or_pad(y, TARGET_LENGTH)

def stats(x):
    x = np.asarray(x)
    return np.concatenate([x.mean(axis=1), x.std(axis=1), x.min(axis=1), x.max(axis=1)])

def extract_features(row):
    y_audio = read_audio(row)
    mfcc = librosa.feature.mfcc(y=y_audio, sr=TARGET_SR, n_mfcc=N_MFCC, n_fft=N_FFT, hop_length=HOP_LENGTH, win_length=WIN_LENGTH)
    delta = librosa.feature.delta(mfcc)
    zcr = librosa.feature.zero_crossing_rate(y_audio, frame_length=WIN_LENGTH, hop_length=HOP_LENGTH)
    rms = librosa.feature.rms(y=y_audio, frame_length=WIN_LENGTH, hop_length=HOP_LENGTH)
    chroma = librosa.feature.chroma_stft(y=y_audio, sr=TARGET_SR, n_fft=N_FFT, hop_length=HOP_LENGTH)
    centroid = librosa.feature.spectral_centroid(y=y_audio, sr=TARGET_SR, n_fft=N_FFT, hop_length=HOP_LENGTH)
    bandwidth = librosa.feature.spectral_bandwidth(y=y_audio, sr=TARGET_SR, n_fft=N_FFT, hop_length=HOP_LENGTH)
    rolloff = librosa.feature.spectral_rolloff(y=y_audio, sr=TARGET_SR, n_fft=N_FFT, hop_length=HOP_LENGTH)
    contrast = librosa.feature.spectral_contrast(y=y_audio, sr=TARGET_SR, n_fft=N_FFT, hop_length=HOP_LENGTH)
    mfcc_vec = np.concatenate([stats(mfcc), stats(delta)])
    full_vec = np.concatenate([stats(mfcc), stats(delta), stats(zcr), stats(rms), stats(chroma), stats(centroid), stats(bandwidth), stats(rolloff), stats(contrast)])
    return mfcc_vec.astype(np.float32), full_vec.astype(np.float32)

cache_path = CACHE_DIR / f'baseline_ml_features_{int(TARGET_DURATION)}s_{len(metadata)}files.npz'
if CACHE_FEATURES and cache_path.exists():
    data = np.load(cache_path)
    X_mfcc = data['X_mfcc']; X_full = data['X_full']; y = data['y']
    print('Loaded feature cache:', cache_path)
else:
    X_mfcc, X_full = [], []
    start = time.time()
    for i, row in enumerate(metadata.itertuples(index=False)):
        a, b = extract_features(row)
        X_mfcc.append(a); X_full.append(b)
        if (i + 1) % 500 == 0:
            print(f'Extracted {i+1}/{len(metadata)} in {(time.time()-start)/60:.1f} min')
    X_mfcc = np.stack(X_mfcc); X_full = np.stack(X_full); y = metadata['label_id'].to_numpy(np.int64)
    if CACHE_FEATURES:
        np.savez_compressed(cache_path, X_mfcc=X_mfcc, X_full=X_full, y=y)
        print('Saved cache:', cache_path)
print('X_mfcc:', X_mfcc.shape, 'X_full:', X_full.shape)


## 5. Experiments and Models


In [ ]:
def split_from_column(col, include_mask=None):
    df = metadata if include_mask is None else metadata[include_mask]
    values = df[col].to_numpy()
    indices = df.index.to_numpy()
    out = {s: indices[values == s] for s in ['train', 'validation', 'test']}
    if len(out['validation']) == 0:
        tr, va = train_test_split(out['train'], test_size=0.12, random_state=SEED, stratify=y[out['train']])
        out['train'], out['validation'] = tr, va
    return out

experiments = []
if RUN_COMBINED_RANDOM:
    experiments.append(('combined_random', split_from_column('split_random')))
if RUN_COMBINED_STRICT:
    strict_mask = metadata['split_strict_project'].isin(['train', 'validation', 'test'])
    experiments.append(('combined_strict_no_tess', split_from_column('split_strict_project', strict_mask)))
if RUN_SINGLE_DATASET:
    for ds in sorted(metadata['dataset'].unique()):
        ds_idx = metadata.index[metadata['dataset'].eq(ds)]
        tr, temp = train_test_split(ds_idx, test_size=0.30, random_state=SEED, stratify=y[ds_idx])
        va, te = train_test_split(temp, test_size=0.50, random_state=SEED, stratify=y[temp])
        experiments.append((f'single_{ds}', {'train': np.array(tr), 'validation': np.array(va), 'test': np.array(te)}))

models = {
    'dummy_most_frequent': DummyClassifier(strategy='most_frequent'),
    'logistic_regression': LogisticRegression(max_iter=1500, class_weight='balanced', n_jobs=-1),
    'linear_svm': LinearSVC(class_weight='balanced', C=1.0, max_iter=5000),
    'random_forest': RandomForestClassifier(n_estimators=350, max_depth=None, class_weight='balanced_subsample', random_state=SEED, n_jobs=-1),
    'extra_trees': ExtraTreesClassifier(n_estimators=500, class_weight='balanced', random_state=SEED, n_jobs=-1),
}
if MODEL_SET == 'full':
    models['rbf_svm'] = SVC(C=10.0, gamma='scale', class_weight='balanced', probability=True)

for name, split_map in experiments:
    print(name, {k: len(v) for k, v in split_map.items()})


## 6. Train, Test and Export


In [ ]:
def eval_predictions(y_true, y_pred):
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'macro_f1': f1_score(y_true, y_pred, average='macro', zero_division=0),
        'weighted_f1': f1_score(y_true, y_pred, average='weighted', zero_division=0),
        'macro_precision': precision_score(y_true, y_pred, average='macro', zero_division=0),
        'macro_recall': recall_score(y_true, y_pred, average='macro', zero_division=0),
    }

def plot_confusion(y_true, y_pred, title, path):
    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(COMMON_EMOTIONS))))
    plt.figure(figsize=(7, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=COMMON_EMOTIONS, yticklabels=COMMON_EMOTIONS)
    plt.title(title); plt.xlabel('Predicted'); plt.ylabel('True'); plt.tight_layout()
    plt.savefig(path, dpi=150); plt.close()

all_metrics, all_per_dataset, all_per_class = [], [], []
feature_sets = {'mfcc': X_mfcc, 'full': X_full}

for exp_name, split_map in experiments:
    for feature_name, X in feature_sets.items():
        for model_name, model in models.items():
            pipe = Pipeline([('scaler', StandardScaler()), ('model', model)])
            start = time.time()
            pipe.fit(X[split_map['train']], y[split_map['train']])
            train_time = time.time() - start
            t0 = time.time()
            pred = pipe.predict(X[split_map['test']])
            inference_time = time.time() - t0
            metric = eval_predictions(y[split_map['test']], pred)
            row = {
                'experiment': exp_name, 'feature_set': feature_name, 'model': model_name,
                'split': 'test', 'n_samples': len(split_map['test']), **metric,
                'train_time_sec': train_time, 'inference_time_sec': inference_time,
                'inference_ms_per_sample': 1000 * inference_time / max(1, len(split_map['test']))
            }
            all_metrics.append(row)
            run_id = f'{exp_name}_{feature_name}_{model_name}'
            joblib.dump(pipe, MODEL_DIR / f'{run_id}.joblib')
            pred_df = metadata.loc[split_map['test'], ['sample_id', 'dataset', 'speaker_id', 'emotion']].copy()
            pred_df['y_true'] = y[split_map['test']]
            pred_df['y_pred'] = pred
            pred_df['pred_emotion'] = [ID_TO_LABEL[i] for i in pred]
            pred_df.to_csv(PRED_DIR / f'predictions_{run_id}.csv', index=False)
            plot_confusion(y[split_map['test']], pred, run_id, FIGURE_DIR / f'confusion_matrix_{run_id}.png')

            report = classification_report(y[split_map['test']], pred, labels=list(range(len(COMMON_EMOTIONS))), target_names=COMMON_EMOTIONS, output_dict=True, zero_division=0)
            pc = pd.DataFrame(report).T.reset_index().rename(columns={'index': 'class'})
            pc.insert(0, 'model', model_name); pc.insert(0, 'feature_set', feature_name); pc.insert(0, 'experiment', exp_name)
            all_per_class.append(pc)

            for ds, part in pred_df.groupby('dataset'):
                all_per_dataset.append({
                    'experiment': exp_name, 'feature_set': feature_name, 'model': model_name, 'dataset': ds, 'n_samples': len(part),
                    **eval_predictions(part['y_true'], part['y_pred'])
                })
            pd.DataFrame(all_metrics).to_csv(REPORT_DIR / 'baseline_per_dataset_metrics.csv', index=False)
            pd.DataFrame(all_per_dataset).to_csv(REPORT_DIR / 'baseline_per_dataset_breakdown.csv', index=False)
            pd.concat(all_per_class, ignore_index=True).to_csv(REPORT_DIR / 'baseline_per_class.csv', index=False)
            print(row)

metrics_df = pd.DataFrame(all_metrics).sort_values('macro_f1', ascending=False)
display_table(metrics_df.head(30))


## 7. Visualize and Package


In [ ]:
metrics_df = pd.read_csv(REPORT_DIR / 'baseline_per_dataset_metrics.csv')
plt.figure(figsize=(12, 5))
sns.barplot(data=metrics_df, x='experiment', y='macro_f1', hue='model')
plt.xticks(rotation=35, ha='right')
plt.title('04 baseline macro-F1 across combined and single-dataset experiments')
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'baseline_per_dataset_macro_f1.png', dpi=160)
plt.show()

summary = {
    'notebook': '04_Baseline_Per_Dataset_Comparison',
    'output_dir': str(OUTPUT_DIR),
    'metrics_csv': str(REPORT_DIR / 'baseline_per_dataset_metrics.csv'),
    'purpose': 'Classical ML baseline for combined random, combined strict and single-dataset comparison.'
}
with open(REPORT_DIR / 'baseline_per_dataset_summary.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

zip_base = PROJECT_ROOT / '04_Baseline_Per_Dataset_outputs_package'
zip_path = shutil.make_archive(str(zip_base), 'zip', OUTPUT_DIR)
print('ZIP:', zip_path)
